In [1]:
import pandas as pd

inventory = pd.read_csv('../data/raw/inventory_data.csv')
vendor = pd.read_csv('../data/raw/vendor_data.csv')
financial = pd.read_csv('../data/raw/financial_data.csv')

print("Inventory shape:", inventory.shape)
print("Vendor shape:", vendor.shape)
print("Financial shape:", financial.shape)

Inventory shape: (500, 11)
Vendor shape: (3, 7)
Financial shape: (500, 4)


In [2]:
print("Inventory columns:")
print(inventory.columns.tolist())
print()
print("Inventory sample:")
inventory.head()

Inventory columns:
['Date', 'Item_ID', 'Item_Type', 'Item_Name', 'Current_Stock', 'Min_Required', 'Max_Capacity', 'Unit_Cost', 'Avg_Usage_Per_Day', 'Restock_Lead_Time', 'Vendor_ID']

Inventory sample:


,Date,Item_ID,Item_Type,Item_Name,Current_Stock,Min_Required,Max_Capacity,Unit_Cost,Avg_Usage_Per_Day,Restock_Lead_Time,Vendor_ID
0,2024-10-01,105,Consumable,Ventilator,1542,264,1018,4467.55,108,17,V001
1,2024-10-02,100,Equipment,Ventilator,2487,656,3556,5832.29,55,12,V001
2,2024-10-03,103,Equipment,Surgical Mask,2371,384,5562,16062.98,470,6,V001
3,2024-10-04,103,Consumable,Surgical Mask,2038,438,1131,744.10,207,15,V002
4,2024-10-05,107,Equipment,IV Drip,2410,338,1013,15426.53,158,12,V003


In [3]:
print("Vendor columns:")
print(vendor.columns.tolist())
vendor.head()

Vendor columns:
['Vendor_ID', 'Vendor_Name', 'Item_Supplied', 'Avg_Lead_Time (days)', 'Cost_Per_Item', 'Last_Order_Date', 'Next_Delivery_Date']


,Vendor_ID,Vendor_Name,Item_Supplied,Avg_Lead_Time (days),Cost_Per_Item,Last_Order_Date,Next_Delivery_Date
0,V001,MedSupplies Inc.,Surgical Mask,5,0.5,2024-09-28,2024-10-03
1,V002,EquipMed Co.,Ventilator,30,20000.0,2024-09-01,2024-10-15
2,V003,HealthTools Ltd.,X-ray Machine,15,5000.0,2024-09-15,2024-10-05


In [4]:
print("Financial columns:")
print(financial.columns.tolist())
financial.head()

Financial columns:
['Date', 'Expense_Category', 'Amount', 'Description']


,Date,Expense_Category,Amount,Description
0,2024-10-01,Staffing,29391.86,Surgical masks
1,2024-10-02,Supplies,47757.71,Surgical masks
2,2024-10-03,Supplies,43996.60,Ventilators
3,2024-10-04,Supplies,27908.42,Surgeons' salaries
4,2024-10-05,Equipment,39719.60,Ventilators


In [5]:
inventory['Days_Until_Stockout'] = inventory['Current_Stock'] / inventory['Avg_Usage_Per_Day']

def get_risk_label(row):
    if row['Days_Until_Stockout'] < row['Restock_Lead_Time']:
        return 'High'
    elif row['Days_Until_Stockout'] < row['Restock_Lead_Time'] * 1.5:
        return 'Medium'
    else:
        return 'Low'

inventory['Risk_Label'] = inventory.apply(get_risk_label, axis=1)
inventory[['Item_Name', 'Current_Stock', 'Avg_Usage_Per_Day', 'Restock_Lead_Time', 'Days_Until_Stockout', 'Risk_Label']].head(10)

,Item_Name,Current_Stock,Avg_Usage_Per_Day,Restock_Lead_Time,Days_Until_Stockout,Risk_Label
0,Ventilator,1542,108,17,14.277778,High
1,Ventilator,2487,55,12,45.218182,Low
2,Surgical Mask,2371,470,6,5.044681,High
3,Surgical Mask,2038,207,15,9.845411,High
4,IV Drip,2410,158,12,15.253165,Medium
5,IV Drip,2258,386,26,5.849741,High
6,IV Drip,387,207,24,1.869565,High
7,IV Drip,123,257,11,0.478599,High
8,Ventilator,790,16,20,49.375000,Low
9,Gloves,2448,418,5,5.856459,Medium


In [6]:
inventory['Risk_Label'].value_counts()

Risk_Label
High      299
Low       156
Medium     45
Name: count, dtype: int64

In [7]:
def get_risk_label(row):
    ratio = row['Days_Until_Stockout'] / row['Restock_Lead_Time']
    if ratio < 1:
        return 'High'      # will run out before restock arrives
    elif ratio < 2:
        return 'Medium'    # cutting it close
    else:
        return 'Low'       # comfortable buffer

inventory['Risk_Label'] = inventory.apply(get_risk_label, axis=1)
inventory['Risk_Label'].value_counts()

Risk_Label
High      299
Low       131
Medium     70
Name: count, dtype: int64

In [8]:
inventory[['Current_Stock', 'Avg_Usage_Per_Day', 'Restock_Lead_Time', 'Days_Until_Stockout']].describe()

,Current_Stock,Avg_Usage_Per_Day,Restock_Lead_Time,Days_Until_Stockout
count,500.000000,500.000000,500.000000,500.000000
mean,2458.644000,261.804000,15.116000,23.937059
std,1390.078133,143.983318,8.610856,58.043229
min,69.000000,2.000000,1.000000,0.168293
25%,1307.750000,150.500000,7.000000,5.454683
50%,2411.500000,257.000000,16.000000,9.305992
75%,3719.000000,392.000000,23.000000,17.678191
max,4976.000000,499.000000,29.000000,598.000000


In [9]:
inventory = inventory.merge(vendor[['Vendor_ID', 'Vendor_Name', 'Avg_Lead_Time (days)', 'Cost_Per_Item']], on='Vendor_ID', how='left')
inventory.head()

,Date,Item_ID,Item_Type,Item_Name,Current_Stock,Min_Required,Max_Capacity,Unit_Cost,Avg_Usage_Per_Day,Restock_Lead_Time,Vendor_ID,Days_Until_Stockout,Risk_Label,Vendor_Name,Avg_Lead_Time (days),Cost_Per_Item
0,2024-10-01,105,Consumable,Ventilator,1542,264,1018,4467.55,108,17,V001,14.277778,High,MedSupplies Inc.,5,0.5
1,2024-10-02,100,Equipment,Ventilator,2487,656,3556,5832.29,55,12,V001,45.218182,Low,MedSupplies Inc.,5,0.5
2,2024-10-03,103,Equipment,Surgical Mask,2371,384,5562,16062.98,470,6,V001,5.044681,High,MedSupplies Inc.,5,0.5
3,2024-10-04,103,Consumable,Surgical Mask,2038,438,1131,744.10,207,15,V002,9.845411,High,EquipMed Co.,30,20000.0
4,2024-10-05,107,Equipment,IV Drip,2410,338,1013,15426.53,158,12,V003,15.253165,Medium,HealthTools Ltd.,15,5000.0


In [10]:
inventory.to_csv('../data/processed/inventory_with_risk.csv', index=False)
print("Saved successfully")

Saved successfully
